# The RemoveMessages Class

### Set the OpenAI API Key as an Environment Variable

In [1]:
%load_ext dotenv
%dotenv
%load_ext mypy_ipython

### Import Relevant Classes and Functions

In [2]:
from langgraph.graph import START, END, StateGraph, add_messages, MessagesState
from typing_extensions import TypedDict
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage, RemoveMessage
from collections.abc import Sequence
from typing import Literal, Annotated

### Get Familiar with RemoveMessages

In [3]:
my_list = add_messages([AIMessage("What is your question?"), 
                        HumanMessage("Could you tell me a grook by Piet Hein?"),
                        AIMessage("Certainly! Here's a well-known grook by Piet Hein..."),
                        AIMessage("Would you like to ask one more question?"),
                        HumanMessage("yes"),
                        AIMessage("What is your question?"),
                        HumanMessage("Where was the poet born?"),
                        AIMessage("Piet Hein was born in Copenhagen, Denmark, on December 16, 1905."),
                        AIMessage("Would you like to ask one more question?")],
                       [HumanMessage("yes")]
                      )

In [4]:
my_list[:-5]

[AIMessage(content='What is your question?', additional_kwargs={}, response_metadata={}, id='60615c39-4ad3-488e-9789-baee54eb2016'),
 HumanMessage(content='Could you tell me a grook by Piet Hein?', additional_kwargs={}, response_metadata={}, id='19943af6-515d-4a9d-bc85-06b6129c9048'),
 AIMessage(content="Certainly! Here's a well-known grook by Piet Hein...", additional_kwargs={}, response_metadata={}, id='0e8dcce1-0b3f-433d-9585-4a64a25ffa29'),
 AIMessage(content='Would you like to ask one more question?', additional_kwargs={}, response_metadata={}, id='dbe8ef50-74bd-443c-b633-aefc88b2af6e'),
 HumanMessage(content='yes', additional_kwargs={}, response_metadata={}, id='1f1d1990-2580-4f74-8e79-d2816b31d9de')]

In [5]:
remove_messages = [RemoveMessage(id = i.id) for i in my_list[:-5]]

In [6]:
remove_messages

[RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='60615c39-4ad3-488e-9789-baee54eb2016'),
 RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='19943af6-515d-4a9d-bc85-06b6129c9048'),
 RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='0e8dcce1-0b3f-433d-9585-4a64a25ffa29'),
 RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='dbe8ef50-74bd-443c-b633-aefc88b2af6e'),
 RemoveMessage(content='', additional_kwargs={}, response_metadata={}, id='1f1d1990-2580-4f74-8e79-d2816b31d9de')]

In [7]:
add_messages(my_list, remove_messages)

[AIMessage(content='What is your question?', additional_kwargs={}, response_metadata={}, id='48802419-d509-4a14-a9b6-17c3cd382f1f'),
 HumanMessage(content='Where was the poet born?', additional_kwargs={}, response_metadata={}, id='ed44ae6d-80d2-4a16-8737-1087d041a220'),
 AIMessage(content='Piet Hein was born in Copenhagen, Denmark, on December 16, 1905.', additional_kwargs={}, response_metadata={}, id='e82108bd-6a0d-4edb-89f7-3e34bde2f67c'),
 AIMessage(content='Would you like to ask one more question?', additional_kwargs={}, response_metadata={}, id='ddf272ea-03a3-425c-8fa4-160418f3af49'),
 HumanMessage(content='yes', additional_kwargs={}, response_metadata={}, id='a5861c23-bdd8-48c3-8ece-0f6f181d5289')]

### Get Familiar with the add_messages Function

In [ ]:
my_list = add_messages([HumanMessage("Hi! I'm Oscar."), 
                        AIMessage("Hey, Oscar. How can I assist you?")],
                       [HumanMessage("Could you summarize today's news?")])

In [ ]:
my_list

### Define the Nodes

In [ ]:
chat = ChatOpenAI(model = "gpt-4o", 
                  seed = 365, 
                  temperature = 0, 
                  max_completion_tokens = 100)

In [ ]:
def ask_question(state: MessagesState) -> MessagesState:
    
    print(f"\n-------> ENTERING ask_question:")
    for i in state["messages"]:
        i.pretty_print()
    
    question = "What is your question?"
    print(question)
    
    return MessagesState(messages = [AIMessage(question), HumanMessage(input())])

In [ ]:
def chatbot(state: MessagesState) -> MessagesState:
    
    print(f"\n-------> ENTERING chatbot:")
    for i in state["messages"]:
        i.pretty_print()
    
    response = chat.invoke(state["messages"])
    response.pretty_print()
    
    return MessagesState(messages = [response])

In [ ]:
def ask_another_question(state: MessagesState) -> MessagesState:
    
    print(f"\n-------> ENTERING ask_another_question:")
    for i in state["messages"]:
        i.pretty_print()
    
    question = "Would you like to ask one more question (yes/no)?"
    print(question)
    
    return MessagesState(messages = [AIMessage(question), HumanMessage(input())])

### Define the Routing Function

In [ ]:
def routing_function(state: MessagesState) -> Literal["ask_question", "__end__"]:
    
    if state["messages"][-1].content == "yes":
        return "ask_question"
    else:
        return "__end__"

### Define the Graph

In [ ]:
graph = StateGraph(MessagesState)

In [ ]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source = "ask_another_question", 
                            path = routing_function)

In [ ]:
graph_compiled = graph.compile()

In [ ]:
graph_compiled

### Test the Graph

In [ ]:
graph_compiled.invoke(MessagesState(messages = []))